In [ ]:
import torch
import copy
import matplotlib.pyplot as plt

from dataset import get_cifar10_dataloaders
from models import get_resnet18, SimpleCNN_NoBN
from trainer import overfit_single_batch, train_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

BATCH_SIZE = 128
OVERFIT_ITERS = 200 
EPOCHS = 15

In [ ]:
trainloader, testloader = get_cifar10_dataloaders(batch_size=BATCH_SIZE)

#base_model = SimpleCNN_NoBN(num_classes=10)
base_model = get_resnet18(num_classes=10)

model_std = copy.deepcopy(base_model)
model_custom = copy.deepcopy(base_model)

In [ ]:
print("=== 1. Стандартное обучение ===")
metrics_std = train_model(model_std, trainloader, testloader, device, 
                          epochs=EPOCHS, start_iteration=0)

print("\n=== 2. Обучение с Batch-Overfit Init ===")
model_custom, warmup_loss = overfit_single_batch(model_custom, trainloader, device, 
                                                 iterations=OVERFIT_ITERS)
print(f"Loss после прогрева: {warmup_loss[-1]:.4f}")

metrics_custom = train_model(model_custom, trainloader, testloader, device, 
                             epochs=EPOCHS, start_iteration=OVERFIT_ITERS)

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(metrics_std['iterations'], metrics_std['train_loss'], marker='o', label='Standard Init')
plt.plot(metrics_custom['iterations'], metrics_custom['train_loss'], marker='s', label='Overfit Init')
plt.title('Train Loss vs Compute Budget')
plt.xlabel('Total Optimizer Steps (Iterations)')
plt.ylabel('Train Loss')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(metrics_std['iterations'], metrics_std['test_acc'], marker='o', label='Standard Init')
plt.plot(metrics_custom['iterations'], metrics_custom['test_acc'], marker='s', label='Overfit Init')
plt.title('Test Accuracy vs Compute Budget')
plt.xlabel('Total Optimizer Steps (Iterations)')
plt.ylabel('Test Accuracy (%)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

plt.tight_layout()
plt.show()